# 02 フィルタと輪郭 — 物体を数える

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 解説（この文章）→ コード → 結果（画像）→ ✅ チェックポイント、の順で進みます。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

**目標**：画像処理の**王道パイプライン**「**グレー → ぼかし → 二値化／エッジ → 輪郭**」を通し、画像の中の**物体を見つけて数える**ところまでを自分のコードで実現する。

> **🎯 このノートのゴール**
>
> - [ ] ぼかし（平滑化）がノイズ低減に効くことを体感する
> - [ ] 二値化（しきい値処理）と**大津の方法**の違いを理解する
> - [ ] `Canny` でエッジを、`findContours` で輪郭を取り出せる
> - [ ] 輪郭の**面積でフィルタ**して、画像内の物体を数えられる

## 2-0-a. なぜ「前処理」をするのか
物体検出や計測の前に、**ノイズを消し、必要な情報だけを際立たせる**のが前処理です。典型はこの 1 本道：

```
元画像(BGR) → グレー(1ch) → ぼかし(平滑化) → 二値化／エッジ → 輪郭抽出
```

各段は「**numpy 配列を別の配列に変換**」しているだけ、という視点を持ち続けてください。

## 2-0. 準備（題材を撮影 + ヘルパー）
**机の上に小物を数個**置いてカメラに向けると、この後の「物体数え」が楽しくなります。
（背景を**白い紙**にして明暗差を上げると、より安定して数えられます）

In [ ]:
import cv2, numpy as np, time
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'  # 日本語ラベルの豆腐対策（要 fonts-noto-cjk）
%matplotlib inline
from picamera2 import Picamera2
def show(im, title='', size=(6,4), gray=False):
    plt.figure(figsize=size)
    plt.imshow(im if gray else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap='gray' if gray else None)
    plt.axis('off')
    if title: plt.title(title)
    plt.show()
picam = Picamera2()
picam.configure(picam.create_preview_configuration(main={'size':(640,480),'format':'RGB888'}))
picam.start(); time.sleep(1)
img = picam.capture_array(); picam.close()
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
show(img, '題材')

## 2-1. ぼかし → 二値化（大津）
ノイズを**ぼかし**で抑え、明るさの**しきい値**で白黒に分けます。

- **ぼかし** `cv2.GaussianBlur`：周囲の画素と平均をとってなめらかに。後段でザラつきを拾わないため。カーネルサイズ `(5,5)` は**奇数**にします（`(3,3)`弱め／`(9,9)`強め。強すぎるとエッジまでボケる）。
- **二値化**：しきい値より明るい画素を白、暗い画素を黒に。**大津の方法**は画像から**しきい値を自動**で決めます。

> **照明が変わるなら「自動」が強い**：固定しきい値は明るさが変わると破綻します。現場では大津や適応的二値化（`cv2.adaptiveThreshold`）のような**自動でしきい値を決める**手法が頼りになります。

In [ ]:
blur = cv2.GaussianBlur(gray, (5,5), 0)
t, bw = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print('大津が選んだしきい値:', t)
show(bw, '二値化（大津）', gray=True)

## 2-2. エッジ検出を**スライダー**で体感
`cv2.Canny` は**明るさが急に変わる境目（エッジ）**を細い線として取り出します。物の**形**を捉えたいときに。

下のセルを実行するとスライダーが出ます。**下しきい値・上しきい値**を動かすと、エッジがリアルタイムに変わります。

> **Canny の 2 つのしきい値**：小さい方を下げると**細かい線まで**拾い、大きい方を上げると**はっきりした線だけ**残ります。まず `(80, 160)` から始め、ノイズが多ければ両方を上げる（比は 1:2〜1:3 が目安）。

In [ ]:
from ipywidgets import interact, IntSlider

@interact(low=IntSlider(80, 0, 255, 5), high=IntSlider(160, 0, 255, 5))
def _(low, high):
    edges = cv2.Canny(blur, low, high)
    plt.figure(figsize=(6,4)); plt.imshow(edges, cmap='gray'); plt.axis('off')
    plt.title(f'Canny(low={low}, high={high})  エッジ画素={int((edges>0).sum())}'); plt.show()

> スライダーが出ない場合は、メニューの **Kernel → Restart** 後にもう一度上から実行してください。

## 2-3. 輪郭抽出で「物体を数える」
二値化画像から**輪郭（つながった白領域の境界）**を取り出し、**小さすぎるノイズを面積で捨て**て、残りを数えます。これが簡単な**物体カウンタ**です。スライダーで `min_area`（数える大きさの下限）を動かせます。

In [ ]:
from ipywidgets import interact, IntSlider
@interact(min_area=IntSlider(500, 0, 5000, 100))
def _(min_area):
    cnts, _ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    objs = [c for c in cnts if cv2.contourArea(c) > min_area]
    out = img.copy()
    for c in objs:
        x,y,w,h = cv2.boundingRect(c)
        cv2.rectangle(out, (x,y), (x+w,y+h), (0,255,0), 2)
    cv2.putText(out, f'objects: {len(objs)}', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
    plt.figure(figsize=(6,4)); plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'検出 {len(objs)} 個 (min_area={min_area})'); plt.show()

✅ **チェックポイント**：枠で囲まれた物体の数が表示されれば OK。`min_area` を動かして、数え方が変わるのを体感しましょう。

> **思った数にならないとき**：背景と物体の**明暗差が小さい**と輪郭が繋がったり切れたりします。`min_area` を上げ下げするか、**背景を白い紙にして対比を上げる**と安定します。これは「現場で照明・背景を整えると AI が楽になる」のミニ版です。

> 🤖 **ChatGPT に聞いてみよう**
> 「OpenCV の `cv2.threshold` の**大津の方法**って何？ なぜしきい値を自分で決めなくていいの？」と聞いて、仕組みを 3 行で理解しよう。エラーが出たら全文を貼って相談。

## ✅ 到達確認

- [ ] 「グレー → ぼかし → 二値化／エッジ → 輪郭」の流れを説明できる
- [ ] 大津の方法が**しきい値を自動で決める**ことを理解した
- [ ] 物体が枠で囲まれ、個数が表示された

---
次は **03_カメラとリアルタイム** で、静止画から**動く映像**へ。